# PAGA on merged PBMC + BAL (subsetted) separated by Condition2

Inputs (converted from Seurat):
- `/group/canc2/anson/working/cf-pbmc-bal/data/SCEs/analysis/pbmc.h5ad`
- `/group/canc2/anson/working/cf-pbmc-bal/data/SCEs/analysis/bal.h5ad`

Steps:
1. Load PBMC and BAL separately
2. Create integer raw count layer `layers['counts']` from `.X` (fails fast if not integer-like)
3. Subset cells by `ann_lv2`
   - BAL: anything starting with `AM` or `MDM`, plus `CD14.mono`
   - PBMC: `CD14.mono` and `CD16.mono`
4. Rename `ann_lv2` to these final levels:
   - `PBMC_CD14.mono`, `PBMC_CD16.mono`, `BAL_CD14.mono`, `BAL_MDM`, `BAL_MDM-PLA2G7hi`, `BAL_AM-CCL`, `BAL_AM`
5. Merge PBMC+BAL into one AnnData
6. Global preprocessing + integration:
   - normalize/log1p, HVG=3000 (seurat_v3), scale
   - PCA, Harmony correction by `sampleID` (if installed)
7. For each Condition2 (C/CFB/CFN): build neighbors on corrected PCs, run PAGA, and plot PAGA only

Plot settings per condition:
- figure size: 5x5
- label font: plain (not bold)
- no border


## Preprocess

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

import matplotlib as mpl
import matplotlib.pyplot as plt

# Optional Harmony
HAVE_HARMONY = False
try:
    import harmonypy as hm
    HAVE_HARMONY = True
except Exception:
    HAVE_HARMONY = False

# Plot defaults
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=600, facecolor='white', fontsize=8)
mpl.rcParams['savefig.dpi'] = 600
mpl.rcParams['savefig.bbox'] = 'tight'


In [ ]:
# --------------------
# Parameters
# --------------------
ROOT = "/group/canc2/anson/working/cf-pbmc-bal"
PBMC_H5AD = os.path.join(ROOT, "data", "SCEs", "analysis", "pbmc.h5ad")
BAL_H5AD  = os.path.join(ROOT, "data", "SCEs", "analysis", "bal.h5ad")

CONDITION_KEY = "Condition2"
BATCH_KEY = "sampleID"
CLUSTER_KEY = "ann_lv2"

COUNTS_LAYER = "counts"

N_HVG = 2000
N_PCS = 20
N_NEIGHBORS = 15

OUTDIR = "paga_pbmc_bal_by_condition2"
os.makedirs(OUTDIR, exist_ok=True)
sc.settings.figdir = OUTDIR


## Helpers: integer counts layer

In [ ]:
def fraction_non_integer_like(X, tol=1e-6, sample_n=200000, seed=0):
    if sp.issparse(X):
        vals = X.data
    else:
        vals = np.asarray(X).ravel()
    if vals.size == 0:
        return 0.0
    rng = np.random.default_rng(seed)
    if vals.size > sample_n:
        vals = vals[rng.choice(vals.size, size=sample_n, replace=False)]
    return float(np.mean(np.abs(vals - np.round(vals)) > tol))


def ensure_counts_layer_from_X(adata: sc.AnnData, layer_key: str = COUNTS_LAYER, tol=1e-6, max_frac_nonint=1e-4):
    if layer_key in adata.layers:
        return
    frac = fraction_non_integer_like(adata.X, tol=tol)
    print(f"fraction non-integer-like in adata.X: {frac:.6g}")
    if frac > max_frac_nonint:
        raise ValueError(
            f"adata.X is not integer-like enough to treat as raw counts (frac_nonint={frac}).\n"
            "Re-convert/export with counts preserved is recommended."
        )
    if sp.issparse(adata.X):
        Xc = adata.X.copy()
        Xc.data = np.rint(Xc.data).astype(np.int32)
        adata.layers[layer_key] = Xc
    else:
        adata.layers[layer_key] = np.rint(np.asarray(adata.X)).astype(np.int32)


def set_counts_as_X(adata: sc.AnnData, layer_key: str = COUNTS_LAYER):
    if layer_key not in adata.layers:
        raise KeyError(f"Missing layer '{layer_key}'")
    adata.X = adata.layers[layer_key].copy()
    return adata


## Load PBMC + BAL

In [ ]:
adata_pbmc = sc.read_h5ad(PBMC_H5AD)
adata_bal = sc.read_h5ad(BAL_H5AD)

for name, ad in [("pbmc", adata_pbmc), ("bal", adata_bal)]:
    for k in [CONDITION_KEY, BATCH_KEY, CLUSTER_KEY]:
        if k not in ad.obs.columns:
            raise KeyError(f"[{name}] Expected obs column '{k}' not found.")
    print(f"{name}: shape={ad.shape} layers={list(ad.layers.keys())}")
    print(f"{name}: X min/max={float(ad.X.min())} {float(ad.X.max())}")

print("Ensuring counts layers...")
ensure_counts_layer_from_X(adata_pbmc, layer_key=COUNTS_LAYER)
ensure_counts_layer_from_X(adata_bal, layer_key=COUNTS_LAYER)
print("pbmc layers:", list(adata_pbmc.layers.keys()))
print("bal layers:", list(adata_bal.layers.keys()))


## Subset + rename ann_lv2

In [ ]:
# --- PBMC subset ---
pbmc_keep = ["CD14.mono", "CD16.mono"]
adata_pbmc_sub = adata_pbmc[adata_pbmc.obs[CLUSTER_KEY].isin(pbmc_keep)].copy()

def rename_pbmc(x: str) -> str:
    if x == "CD14.mono":
        return "PBMC_CD14.mono"
    if x == "CD16.mono":
        return "PBMC_CD16.mono"
    return None

adata_pbmc_sub.obs[CLUSTER_KEY] = adata_pbmc_sub.obs[CLUSTER_KEY].astype(str).map(rename_pbmc)
adata_pbmc_sub = adata_pbmc_sub[adata_pbmc_sub.obs[CLUSTER_KEY].notna()].copy()

# --- BAL subset ---
bal_lv2 = adata_bal.obs[CLUSTER_KEY].astype(str)
bal_keep_mask = bal_lv2.str.startswith("AM") | bal_lv2.str.startswith("MDM") | bal_lv2.eq("CD14.mono")
adata_bal_sub = adata_bal[bal_keep_mask].copy()

def rename_bal(x: str) -> str:
    # CD14.mono
    if x == "CD14.mono":
        return "BAL_CD14.mono"
    # AM
    if x == "AM-CCL":
        return "BAL_AM-CCL"
    if x.startswith("AM"):
        return "BAL_AM"
    # MDM
    if x == "MDM-PLA2G7hi":
        return "BAL_MDM-PLA2G7hi"
    if x.startswith("MDM"):
        return "BAL_MDM"
    return None

adata_bal_sub.obs[CLUSTER_KEY] = adata_bal_sub.obs[CLUSTER_KEY].astype(str).map(rename_bal)
adata_bal_sub = adata_bal_sub[adata_bal_sub.obs[CLUSTER_KEY].notna()].copy()

print("PBMC subset counts:")
print(adata_pbmc_sub.obs[CLUSTER_KEY].value_counts())
print("BAL subset counts:")
print(adata_bal_sub.obs[CLUSTER_KEY].value_counts())


In [ ]:
MAX_PER_GROUP = 4177
RANDOM_SEED = 0

def downsample_per_group(ad, group_key=CLUSTER_KEY, max_n=MAX_PER_GROUP, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    keep = []
    for g, idx in ad.obs.groupby(group_key).indices.items():
        idx = np.array(idx)
        if len(idx) > max_n:
            idx = rng.choice(idx, size=max_n, replace=False)
        keep.append(idx)
    keep = np.concatenate(keep)
    return ad[keep].copy()

adata_pbmc_sub = downsample_per_group(adata_pbmc_sub, CLUSTER_KEY, MAX_PER_GROUP)
adata_bal_sub  = downsample_per_group(adata_bal_sub,  CLUSTER_KEY, MAX_PER_GROUP)

print("PBMC subset counts after downsampling:")
print(adata_pbmc_sub.obs[CLUSTER_KEY].value_counts())
print("BAL subset counts after downsampling:")
print(adata_bal_sub.obs[CLUSTER_KEY].value_counts())

## Merge PBMC + BAL subsets

Concatenate with an outer join on genes. Downstream HVG selection will pick a shared set.

In [ ]:
# Keep counts layer + set as X for concatenation
set_counts_as_X(adata_pbmc_sub, COUNTS_LAYER)
set_counts_as_X(adata_bal_sub, COUNTS_LAYER)

# Concatenate (mapping keys already define dataset labels; do NOT pass keys=...)
adata = sc.concat(
    {"PBMC": adata_pbmc_sub, "BAL": adata_bal_sub},
    axis=0,
    join="outer",
    label="dataset",
    fill_value=0,
    index_unique=None
)

# Make obs names unique (avoids the warning you saw)
adata.obs_names_make_unique()

# Ensure counts layer exists post-concat
if COUNTS_LAYER not in adata.layers:
    # If concat dropped layers, reconstruct counts from X (should still be counts)
    ensure_counts_layer_from_X(adata, layer_key=COUNTS_LAYER)

# Set ann_lv2 as categorical with an automatically derived, stable ordering:
# PBMC_* first, then BAL_*; otherwise alphabetical.
levels = sorted(adata.obs[CLUSTER_KEY].astype(str).unique().tolist())
levels = sorted(levels, key=lambda s: (0 if s.startswith("PBMC_") else 1, s))
adata.obs[CLUSTER_KEY] = pd.Categorical(adata.obs[CLUSTER_KEY].astype(str), categories=levels, ordered=True)

print("Merged shape:", adata.shape)
print("Merged ann_lv2 levels:", adata.obs[CLUSTER_KEY].cat.categories.tolist())
print("Merged ann_lv2 counts:\n", adata.obs[CLUSTER_KEY].value_counts())
print("layers after concat:", list(adata.layers.keys()))

In [ ]:
# desired order
levels = [
    "PBMC_CD14.mono",
    "PBMC_CD16.mono",
    "BAL_CD14.mono",
    "BAL_MDM",
    "BAL_MDM-PLA2G7hi",
    "BAL_AM",
    "BAL_AM-CCL",
]

# sanity checks
present = pd.Series(adata.obs[CLUSTER_KEY].astype(str).unique())
missing = [x for x in levels if x not in set(present)]
extra   = sorted(set(present) - set(levels))

if missing:
    print("WARNING: these expected ann_lv2 levels are missing from adata:", missing)
if extra:
    print("WARNING: these ann_lv2 levels are present but not in your specified order list:", extra)

# keep all present categories; put unknown ones at the end (optional but safe)
final_levels = [x for x in levels if x in set(present)] + [x for x in extra]

adata.obs[CLUSTER_KEY] = pd.Categorical(
    adata.obs[CLUSTER_KEY].astype(str),
    categories=final_levels,
    ordered=True,
)

print("Merged ann_lv2 levels:", adata.obs[CLUSTER_KEY].cat.categories.tolist())

## Global preprocessing + integration (shared HVGs/PCs)

log normalize + scale, HVG=3000 (Seurat vst via scanpy seurat_v3), PCA, Harmony correction by donor (`sampleID`).

In [ ]:
# Start from counts
adata.X = adata.layers[COUNTS_LAYER].copy()

# Normalize + log
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# HVGs
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="cell_ranger")
adata = adata[:, adata.var["highly_variable"]].copy()

# Scale
sc.pp.scale(adata)

# PCA
sc.tl.pca(adata, n_comps=50, svd_solver="arpack")

# Harmony correction on PCs
rep = "X_pca"
if HAVE_HARMONY:
    Z = adata.obsm["X_pca"]  # (n_cells, n_pcs)
    meta = adata.obs[[BATCH_KEY]].copy()

    ho = hm.run_harmony(Z, meta, vars_use=[BATCH_KEY], max_iter_harmony=30)

    Zcorr = ho.Z_corr
    print("Raw Harmony output shape:", np.shape(Zcorr))

    # Ensure 2D numpy array
    Zcorr = np.asarray(Zcorr)
    if Zcorr.ndim != 2:
        raise ValueError(f"Harmony returned non-2D array: shape={Zcorr.shape}")

    # Make it (n_cells, n_pcs)
    if Zcorr.shape[0] == adata.n_obs:
        Xh = Zcorr
    elif Zcorr.shape[1] == adata.n_obs:
        Xh = Zcorr.T
    else:
        raise ValueError(
            f"Cannot align Harmony output shape {Zcorr.shape} with n_obs={adata.n_obs}"
        )

    adata.obsm["X_pca_harmony"] = Xh
    rep = "X_pca_harmony"
    print("Harmony corrected PCs stored in X_pca_harmony:", adata.obsm[rep].shape)
else:
    print("harmonypy not installed; using uncorrected PCs")

print("Using representation:", rep, "shape:", adata.obsm[rep].shape)


## PAGA per Condition2 (PAGA plot only)


In [ ]:
# PAGA: no node labels on graph, legend on the right (Carto Vivid palette),
# plus a combined figure with 3 PAGA panels in order: C, CFB, CFN.

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

os.makedirs(OUTDIR, exist_ok=True)

# ---------- config ----------
ann_lv2_order = [
    "PBMC_CD14.mono",
    "PBMC_CD16.mono",
    "BAL_CD14.mono",
    "BAL_MDM",
    "BAL_MDM-PLA2G7hi",
    "BAL_AM",
    "BAL_AM-CCL",
]

# carto_pal(12, "Vivid")
VIVID_12 = [
    "#E58606", "#5D69B1", "#52BCA3", "#99C945", "#CC61B0", "#24796C",
    "#DAA51B", "#2F8AC4", "#764E9F", "#ED645A", "#A5AA99", "#1D6996"
]

# stable mapping (category -> color)
paga_color_dict = {cat: VIVID_12[i] for i, cat in enumerate(ann_lv2_order)}

def apply_cluster_order_and_colors(ad, key=CLUSTER_KEY):
    # enforce categorical order (keeps absent levels)
    ad.obs[key] = pd.Categorical(
        ad.obs[key].astype(str),
        categories=ann_lv2_order,
        ordered=True,
    )
    # palette list aligned to categories (what scanpy uses internally)
    ad.uns[f"{key}_colors"] = [paga_color_dict[c] for c in ad.obs[key].cat.categories]

def build_legend_handles(ad, key=CLUSTER_KEY):
    cats = ad.obs[key].cat.categories.tolist()
    cols = ad.uns.get(f"{key}_colors", None)
    if cols is None:
        raise KeyError(f"Missing ad.uns['{key}_colors']; call apply_cluster_order_and_colors() first.")

    handles = [
        Line2D([0], [0], marker="o", linestyle="None",
               markerfacecolor=cols[i], markeredgecolor="none",
               markersize=8, label=str(cats[i]))
        for i in range(len(cats))
    ]
    return handles

def _remove_scanpy_texts_keep_title(ax, title):
    # remove node labels (scanpy draws them as ax.texts)
    for t in list(ax.texts):
        t.remove()
    ax.set_title(title, fontweight="normal")

def paga_draw_on_ax(ad, ax, cond, threshold=0.05):
    # compute graph for this subset
    sc.pp.neighbors(
        ad,
        n_neighbors=N_NEIGHBORS,
        n_pcs=N_PCS,
        use_rep=rep,
    )
    sc.tl.paga(ad, groups=CLUSTER_KEY)

    ax.set_axis_off()

    sc.pl.paga(
        ad,
        threshold=threshold,
        layout="fr",
        title=f"{cond}: PAGA",
        show=False,
        use_raw=False,
        edge_width_scale=0.5,
        color=CLUSTER_KEY,  # uses ad.uns[f"{CLUSTER_KEY}_colors"]
        ax=ax,
    )

    _remove_scanpy_texts_keep_title(ax, f"{cond}: PAGA")

def paga_plot_single_with_legend(ad, cond, threshold=0.05):
    # Make sure ordering/palette are set on THIS object
    apply_cluster_order_and_colors(ad, CLUSTER_KEY)

    fig, ax = plt.subplots(figsize=(5, 5))  # extra width for legend
    paga_draw_on_ax(ad, ax, cond, threshold=threshold)

    handles = build_legend_handles(ad, CLUSTER_KEY)
    ax.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False,
        borderaxespad=0.0,
        fontsize=9,
        handletextpad=0.4,
        labelspacing=0.4,
    )

    out = os.path.join(OUTDIR, f"paga_{cond}_tree.jpeg")
    plt.savefig(out, dpi=1200, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out)

def paga_plot_triptych(adatas_by_cond, cond_order=("C", "CFB", "CFN"), threshold=0.05):
    # combined plot: 3 panels left->right, one shared legend on the right
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=False)

    # Draw each panel
    for ax, cond in zip(axes, cond_order):
        ad = adatas_by_cond[cond]
        apply_cluster_order_and_colors(ad, CLUSTER_KEY)
        paga_draw_on_ax(ad, ax, cond, threshold=threshold)

    # One shared legend (use first condition's categories/colors; they are the same order)
    handles = build_legend_handles(adatas_by_cond[cond_order[0]], CLUSTER_KEY)
    fig.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=False,
        fontsize=9,
    )

    out = os.path.join(OUTDIR, "paga_C_CFB_CFN_triptych.jpeg")
    plt.savefig(out, dpi=1200, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out)

# ----------------------------
# Run: make per-condition objects in desired order and plot
# ----------------------------
cond_order = ["C", "CFB", "CFN"]

adatas_by_cond = {}
for cond in cond_order:
    ad_cond = adata[adata.obs[CONDITION_KEY].astype(str) == cond].copy()

    # ensure category order is retained even if levels are absent in this condition
    ad_cond.obs[CLUSTER_KEY] = pd.Categorical(
        ad_cond.obs[CLUSTER_KEY].astype(str),
        categories=ann_lv2_order,
        ordered=True,
    )
    adatas_by_cond[cond] = ad_cond

# 1) separate plots (each has its own legend)
for cond in cond_order:
    paga_plot_single_with_legend(adatas_by_cond[cond], cond, threshold=0.05)

# 2) combined 3-panel plot (C, then CFB, then CFN)
paga_plot_triptych(adatas_by_cond, cond_order=tuple(cond_order), threshold=0.05)